# Self-Serve Google Sheets Ingestion

## Project Overview
Operations and business teams keep their working data where they work: Google Sheets. Analytics needs that data in the lake — typed, versioned, and change-aware — without asking the team to adopt a new tool, and without an engineer in the loop for every new sheet.

This notebook is the ingestion pattern I used in production, reduced to its two durable ideas:

1. **Self-serve reads** — any analyst can point one function at a sheet URL and get a clean, typed DataFrame back.
2. **Change capture** — the Sheets API tells you *what a sheet looks like now*, not *what changed*. Snapshot diffing turns consecutive pulls into an insert/update/delete changelog you can trust downstream.

Everything downstream of these two steps (warehouse DDL, dashboards) belongs to the modern stack — land the raw data, hand off to dbt. The last section shows that hand-off.

> **Note on data:** `data/` bundles two consecutive "API pulls" of an anonymized ops outreach sheet, generated by `data/generate_snapshots.py` with known ground truth: day 2 contains exactly 3 inserts, 2 updates, and 1 delete. Live-API cells are guarded so the notebook runs end-to-end without credentials.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd
import numpy as np

pd.options.display.max_columns = None

LIVE = Path("client_secret.json").exists()   # real creds present? then hit the API
print(f"live API mode: {LIVE} — using bundled snapshots" if not LIVE else "live API mode")

live API mode: False — using bundled snapshots


# 1. Reading a Sheet Like an Engineer

The Sheets API is simple; production reads around it are not. Three things the wrapper below handles:

- **Auth that survives a scheduler.** OAuth 2.0 with a cached refresh token: one interactive consent, then every scheduled run refreshes silently. Scope is `spreadsheets` only — the job can read and write sheets, nothing else in Drive.
- **Ragged rows.** `values().get` trims trailing empty cells per row, so row lengths vary. Pad to the header width before building a frame.
- **Everything is a string.** The API returns what the user sees. Typing is *our* job, and pretending otherwise is how `$98,410.00` ends up summed as zero.

In [2]:
def read_sheet(sheet_url: str, sheet_range: str) -> list[list[str]]:
    """Pull a range from a Google Sheet. One consent, then non-interactive."""
    import pickle
    from google.auth.transport.requests import Request
    from google_auth_oauthlib.flow import InstalledAppFlow
    from googleapiclient.discovery import build

    SCOPES = ["https://www.googleapis.com/auth/spreadsheets"]
    sheet_id = sheet_url.split("/")[5]

    creds = None
    if Path("token.pickle").exists():
        creds = pickle.loads(Path("token.pickle").read_bytes())
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("client_secret.json", SCOPES)
            creds = flow.run_local_server(port=0)
        Path("token.pickle").write_bytes(pickle.dumps(creds))

    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=sheet_id, range=sheet_range
    ).execute()
    return result.get("values", [])


def pull(snapshot_date: str) -> list[list[str]]:
    """Live pull when credentials exist; otherwise the bundled capture."""
    if LIVE:
        return read_sheet("https://docs.google.com/spreadsheets/d/EXAMPLE_ID/edit",
                          "Operations Support Campaign")
    raw = pd.read_csv(f"data/snapshot_{snapshot_date}.csv", header=None, dtype=str,
                      keep_default_na=False)
    return raw.values.tolist()


values = pull("2022-08-08")
print(f"{len(values)} rows pulled; first row is the header: {values[0][:4]} ...")

41 rows pulled; first row is the header: ['Record ID', 'Merchant ID', 'Contact Date', 'Rep'] ...


# 2. Normalization: Undoing the Spreadsheet

One function turns an API payload into an analytics-grade frame. Every line answers a real sheet pathology: ragged rows, header drift (`Record ID` vs `record id `), human-formatted currency, checkbox strings, and cleared-cell empties.

In [3]:
def normalize(values: list[list[str]]) -> pd.DataFrame:
    """API payload -> typed DataFrame with stable snake_case columns."""
    header, *rows = values
    width = len(header)
    rows = [r + [""] * (width - len(r)) for r in rows]          # pad ragged rows

    df = pd.DataFrame(rows, columns=header)
    df.columns = (df.columns.str.strip().str.lower()
                    .str.replace(r"\s+", "_", regex=True))      # header drift
    df = df.replace("", pd.NA)                                   # cleared cells

    df["contact_date"] = pd.to_datetime(df["contact_date"], errors="coerce")
    df["monthly_volume_usd"] = (df["monthly_volume_usd"]
                                .str.replace(r"[$,]", "", regex=True)
                                .astype(float))                  # $98,410.00 -> 98410.0
    df["follow_up_needed"] = df["follow_up_needed"].map({"TRUE": True, "FALSE": False})
    return df


day1 = normalize(pull("2022-08-08"))
day1.dtypes

record_id                     object
merchant_id                   object
contact_date          datetime64[ns]
rep                           object
contact_type                  object
call_outcome                  object
follow_up_needed                bool
monthly_volume_usd           float64
notes                         object
dtype: object

In [4]:
day1.head(8)

,record_id,merchant_id,contact_date,rep,contact_type,call_outcome,follow_up_needed,monthly_volume_usd,notes
0,rec-0001,m26b563b1,2022-08-02,j.ibarra,email,escalated,True,238208.72,duplicate account
1,rec-0002,m1a6e72b9,2022-08-06,s.okafor,email,left voicemail,False,107424.99,<NA>
2,rec-0003,m55e7d67e,2022-08-07,a.moss,email,left voicemail,False,53180.32,payor dispute
3,rec-0004,m08fc2081,2022-08-02,a.moss,email,no answer,False,54891.57,onboarding question
4,rec-0005,m9fbba638,2022-08-01,j.ibarra,email,escalated,True,181744.26,<NA>
5,rec-0006,m9e1fcc46,2022-08-05,k.tran,call,<NA>,False,4202.80,requested W-9
6,rec-0007,m6ed299e4,2022-08-03,s.okafor,call,escalated,False,242609.64,<NA>
7,rec-0008,m9b914a48,2022-08-04,s.okafor,email,escalated,True,59526.12,payor dispute


The dtype report is the point: dates are datetimes, currency is float, checkboxes are booleans. A `SUM(monthly_volume_usd)` downstream now means something.

# 3. Capturing Changes Through the API

The Sheets API has no change feed. Three strategies, honestly compared:

| Strategy | How | Verdict |
| --- | --- | --- |
| **Snapshot diffing** | pull the full range on a schedule, diff against the previous pull | chosen — sheets are small, the diff is exact (deletes included), and the loader stays stateless beyond one stored snapshot |
| Drive `revisions` API | file-level revision history | tells you *that* the file changed, not *which rows* — a trigger signal, not CDC |
| Apps Script `onEdit` triggers | push cell edits to a webhook | true realtime, but code living inside the sheet, per sheet — the opposite of self-serve |

Diffing is exact because every record hashes to one value: any cell change flips the row hash. Keyed on `record_id`, set arithmetic does the rest.

In [5]:
def row_hash(df: pd.DataFrame, key: str = "record_id") -> pd.Series:
    """One stable hash per row over all non-key columns."""
    payload = df.drop(columns=[key]).astype(str).agg("|".join, axis=1)
    return payload.map(lambda s: hashlib.md5(s.encode()).hexdigest())


def capture_changes(prev: pd.DataFrame, curr: pd.DataFrame,
                    key: str = "record_id") -> pd.DataFrame:
    """Classify every record as insert / update / delete between two pulls."""
    p = prev.assign(_hash=row_hash(prev)).set_index(key)
    c = curr.assign(_hash=row_hash(curr)).set_index(key)

    inserted = c.index.difference(p.index)
    deleted  = p.index.difference(c.index)
    common   = c.index.intersection(p.index)
    updated  = common[c.loc[common, "_hash"] != p.loc[common, "_hash"]]

    changelog = pd.concat([
        c.loc[inserted].assign(change_type="insert"),
        c.loc[updated].assign(change_type="update"),
        p.loc[deleted].assign(change_type="delete"),
    ]).drop(columns="_hash").reset_index()
    return changelog


day2 = normalize(pull("2022-08-09"))
changelog = capture_changes(day1, day2)
changelog["change_type"].value_counts()

change_type
insert    3
update    2
delete    1
Name: count, dtype: int64

In [6]:
changelog.sort_values(["change_type", "record_id"])

,record_id,merchant_id,contact_date,rep,contact_type,call_outcome,follow_up_needed,monthly_volume_usd,notes,change_type
5,rec-0030,mf94d2198,2022-08-06,j.ibarra,email,resolved,False,64975.75,duplicate account,delete
0,rec-0041,mc3376f26,2022-08-02,k.tran,email,no answer,False,216246.35,<NA>,insert
1,rec-0042,m279cd2f4,2022-08-03,j.ibarra,call,no answer,True,229330.08,onboarding question,insert
2,rec-0043,m0062f1c5,2022-08-05,s.okafor,call,no answer,True,188929.03,<NA>,insert
3,rec-0005,m9fbba638,2022-08-01,j.ibarra,email,resolved,False,181744.26,<NA>,update
4,rec-0018,m8ffefe4e,2022-08-07,j.ibarra,call,left voicemail,True,98410.00,<NA>,update


Ground truth for the bundled snapshots is 3 inserts, 2 updates, 1 delete — and that is exactly what the diff recovered, including the delete, which most "just append the new pull" pipelines silently lose.

The two updates show *why* row-hashing beats column-watching: one was an outcome filled in (`no answer` → `resolved`), the other a corrected volume. No schema knowledge required — any cell change is caught.

# 4. Landing It — and Handing Off to dbt

The loader's job ends at the raw layer: current snapshot plus append-only changelog, partitioned by pull date. In the original 2022 build, what followed was hand-written Athena DDL, a Parquet CTAS conversion, and QuickSight — all of which today collapses into *land Parquet, let dbt own the rest* (staging model, tests mirroring the normalization rules, snapshots for history).

In [7]:
lake = Path("lake")
(lake / "raw_ops_outreach").mkdir(parents=True, exist_ok=True)
(lake / "changelog_ops_outreach").mkdir(parents=True, exist_ok=True)

day2.to_parquet(lake / "raw_ops_outreach" / "pull_date=2022-08-09.parquet", index=False)
changelog.assign(pulled_at="2022-08-09").to_parquet(
    lake / "changelog_ops_outreach" / "pull_date=2022-08-09.parquet", index=False)

sorted(str(p) for p in lake.rglob("*.parquet"))

['lake/changelog_ops_outreach/pull_date=2022-08-09.parquet',
 'lake/raw_ops_outreach/pull_date=2022-08-09.parquet']

In [8]:
import duckdb

# what dbt's staging model sees — typed columns, straight off the raw layer
duckdb.sql("""
    SELECT
        contact_type,
        COUNT(*)                       AS contacts,
        SUM(monthly_volume_usd)        AS volume_usd,
        SUM(CASE WHEN follow_up_needed THEN 1 ELSE 0 END) AS follow_ups
    FROM 'lake/raw_ops_outreach/*.parquet'
    GROUP BY 1
    ORDER BY 2 DESC
""").df()

,contact_type,contacts,volume_usd,follow_ups
0,email,21,2861586.88,12.0
1,call,21,2266874.26,13.0


# Conclusions

- **Self-serve is a function signature.** `normalize(pull(url))` is the whole analyst-facing API — no tickets, no engineer in the loop for sheet number twelve.
- **CDC on Sheets is snapshot diffing.** The API reports state, not changes; a row hash and set arithmetic recover exact inserts, updates, *and deletes* with one snapshot of state.
- **Stop at the raw layer.** Typed Parquet plus a changelog is the right seam: everything after it — models, tests, history — is dbt's job now, and the 2022 version's hand DDL and dashboard wiring is the part that aged out.